# Brent Pattern System - Arranque en Colab

Este notebook está pensado para el primer paso del proyecto: subir la librería, validar el bundle, lanzar una corrida corta de entrenamiento/inferencia y calcular métricas de paper.

## 1. Opcional: montar Google Drive
Activa esta celda si vas a trabajar con ficheros guardados en Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Sube estos ficheros a Colab

Necesitas como mínimo:
- brent_pattern_system_series_downloader_adapted.zip
- dataset_wide_with_target.csv
- dataset_wide_features_zscore.csv
- feature_means.csv
- feature_stds.csv
- X_train.npy, X_test.npy, y_train.npy, y_test.npy
- paper_metrics.py

In [ ]:
from google.colab import files
uploaded = files.upload()

## 3. Configuración base de rutas

In [ ]:
from pathlib import Path
import json, shutil, zipfile, os

PROJECT_ZIP = Path('/content/brent_pattern_system_series_downloader_adapted.zip')
DATASET_DIR = Path('/content')
PROJECT_DIR = Path('/content/brent_project')
ARTIFACTS_DIR = Path('/content/artifacts_colab_run1')
BACKEND = 'torch'   # 'torch' o 'tensorflow'

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PROJECT_ZIP, 'r') as zf:
    zf.extractall(PROJECT_DIR)

print('Proyecto descomprimido en', PROJECT_DIR)
print('Dataset dir =', DATASET_DIR)
print('Artifacts dir =', ARTIFACTS_DIR)

## 4. Instalar dependencias
Para la primera corrida en Colab recomiendo **PyTorch solo**. TensorFlow se deja para una segunda pasada.

In [ ]:
%cd /content/brent_project
!python -m pip install --upgrade pip
!pip install -U numpy pandas scikit-learn timm
!python - <<'PY'
import importlib, subprocess, sys
for pkg in ['torch', 'torchvision']:
    try:
        m = importlib.import_module(pkg)
        print(pkg, m.__version__)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
PY
# Si luego quieres probar TensorFlow, ejecuta también: pip install tensorflow

## 5. Crear config de ejecución de Colab
Se parte del example_config_attached_bundle.json y se reescriben rutas y parámetros de la corrida.

In [ ]:
from pathlib import Path
import json

base_cfg_path = PROJECT_DIR / 'brent_pattern_system' / 'example_config_attached_bundle.json'
with open(base_cfg_path, 'r', encoding='utf-8') as fh:
    cfg = json.load(fh)

cfg['data']['series_bundle']['dataset_dir'] = str(DATASET_DIR)
cfg['training']['device'] = 'cuda'
cfg['training']['epochs'] = 3
cfg['training']['fine_tune_epochs'] = 2
cfg['training']['batch_size'] = 16
cfg['training']['pretrained_backbone'] = True
cfg['dataset']['use_synthetic_pretrain'] = False
cfg['output']['root_dir'] = str(ARTIFACTS_DIR)
cfg['output']['model_dir'] = str(ARTIFACTS_DIR / 'models')
cfg['output']['report_dir'] = str(ARTIFACTS_DIR / 'reports')
cfg['output']['dataset_dir'] = str(ARTIFACTS_DIR / 'datasets')
cfg['output']['metadata_dir'] = str(ARTIFACTS_DIR / 'metadata')

run_cfg_path = PROJECT_DIR / 'run_config_colab.json'
with open(run_cfg_path, 'w', encoding='utf-8') as fh:
    json.dump(cfg, fh, indent=2, ensure_ascii=False)

print(run_cfg_path)
print(json.dumps(cfg, indent=2, ensure_ascii=False)[:1200])

## 6. Validación y perfilado del bundle
Si esto falla, no tiene sentido entrenar todavía.

%cd /content/brent_project
!python -m brent_pattern_system.validate_bundle --config /content/brent_project/run_config_colab.json
!python -m brent_pattern_system.profile_bundle --config /content/brent_project/run_config_colab.json

## 7. Entrenamiento
Primera corrida corta para smoke test. Cuando todo funcione, sube epochs y fine_tune_epochs.

%cd /content/brent_project
!python -m brent_pattern_system.train_torch --config /content/brent_project/run_config_colab.json

## 8. Inferencia y outliers

%cd /content/brent_project
!python -m brent_pattern_system.inference --config /content/brent_project/run_config_colab.json --backend torch

## 9. Script de métricas de paper
Genera resumen JSON, CSV por split, matriz de confusión, curvas de entrenamiento y tabla LaTeX.

%cd /content
!python /content/paper_metrics.py --artifacts-root /content/artifacts_colab_run1 --backend torch

## 10. Inspección rápida de resultados
Estas tablas son las primeras que debes mirar antes de seguir.

In [ ]:
import pandas as pd
from pathlib import Path
metrics_dir = Path('/content/artifacts_colab_run1/reports/torch_paper_metrics')
display(pd.read_csv(metrics_dir / 'metrics_by_split.csv'))
display(pd.read_csv('/content/artifacts_colab_run1/reports/torch_prediction_report.csv').head(10))
display(pd.read_csv('/content/artifacts_colab_run1/reports/torch_outlier_report.csv').head(10))

## 11. Descargar artefactos
Descarga al menos la carpeta de métricas y los reportes de predicción/outliers.

In [ ]:
%cd /content
!zip -r brent_colab_outputs.zip artifacts_colab_run1 >/dev/null
from google.colab import files
files.download('/content/brent_colab_outputs.zip')